# Subperiod Estimation: 1984-2000 vs 2001-2017

This notebook estimates the LASSO model on two distinct subperiods:
1. **Period 1:** 1984-01-01 to 2000-12-31
2. **Period 2:** 2001-01-01 to 2017-12-31

For each subperiod, we independently tune the hyperparameters (λ and window size) using Optuna, searching for the optimal CMCE-aligned score. We then perform a full cross-sectional estimation using the tuned hyperparameters on the respective subperiod data, outputting separate `.h5` tensor datasets and `.csv` summaries.

In [23]:
import sys, os, warnings, random, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import optuna
from joblib import Parallel, delayed

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Paths ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path('.').resolve()
REPO_ROOT    = NOTEBOOK_DIR.parent.parent
SCRIPTS_DIR  = REPO_ROOT / 'Empirical' / 'scripts'
DATA_DIR     = REPO_ROOT / 'Data'
OUT_DIR      = REPO_ROOT / 'Results' / 'Estimation' / 'Subperiods'

OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SCRIPTS_DIR))

from grid_search import estimate_single_config_fast
from stage2 import compute_alm_returns

# ── Globals & Hyperparameters ────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

TUNE_FRAC    = 0.20     # Fraction of stocks per subperiod for tuning
N_TRIALS     = 200      # Optuna trials per subperiod
N_LAGS       = 1
TARGET_VOL   = 0.0175   # Cross-sectional median daily return std
LAMBDA_LOW, LAMBDA_HIGH = 1e-3, 0.1
WINDOW_LOW, WINDOW_HIGH = 60,   400

N_JOBS       = -1
BATCH_SIZE   = 50

print('Setup complete. Output directory:', OUT_DIR)

Setup complete. Output directory: C:\Users\jonat\Lasso_paper\Results\Estimation\Subperiods


In [24]:
# ── Data Loading ─────────────────────────────────────────────────────────────
X_all = pd.read_csv(DATA_DIR / 'clean_data' / 'final_topic_only_features.csv',
                    index_col=0, parse_dates=True)
R_all = pd.read_csv(DATA_DIR / 'clean_data' / 'daily_stock_data_filtered.csv',
                    index_col=0, parse_dates=True)

common_idx = X_all.index.intersection(R_all.index)
X_all = X_all.loc[common_idx]
R_all = R_all.loc[common_idx]
X_all.columns = [str(c).replace(' ', '_') for c in X_all.columns]

print(f'Features : {X_all.shape}')
print(f'Returns  : {R_all.shape}')
print(f'Common   : {len(common_idx)} days  [{common_idx[0].date()} to {common_idx[-1].date()}]')

Features : (8431, 180)
Returns  : (17997255, 6)
Common   : 8431 days  [1984-01-16 to 2017-06-30]


In [25]:
r_all_wide = (
    R_all
    .reset_index()                         # bring 'date' back as a column
    .pivot(index='date', columns='permno', values='return')
)

r_all_wide.index.name = 'date'
r_all_wide.columns.name = None             # clean up the column axis label

print(r_all_wide.shape)
print(r_all_wide.head())

(8431, 4711)
            10051     10057  10064     10065  10071  10085  10092  10108  \
date                                                                       
1984-01-16    NaN -0.032110    NaN  0.020548    NaN    NaN    NaN    NaN   
1984-01-17    NaN -0.004739    NaN  0.000000    NaN    NaN    NaN    NaN   
1984-01-18    NaN  0.014286    NaN  0.007407    NaN    NaN    NaN    NaN   
1984-01-19    NaN  0.004695    NaN  0.000000    NaN    NaN    NaN    NaN   
1984-01-20    NaN  0.000000    NaN -0.007353    NaN    NaN    NaN    NaN   

            10119  10120  ...  93406  93415  93416  93418  93419  93420  \
date                      ...                                             
1984-01-16    NaN    NaN  ...    NaN    NaN    NaN    NaN    NaN    NaN   
1984-01-17    NaN    NaN  ...    NaN    NaN    NaN    NaN    NaN    NaN   
1984-01-18    NaN    NaN  ...    NaN    NaN    NaN    NaN    NaN    NaN   
1984-01-19    NaN    NaN  ...    NaN    NaN    NaN    NaN    NaN    NaN   
1984

In [26]:
# ── Subperiods Definition ────────────────────────────────────────────────────
SUBPERIODS = {
    'Period_1': ('1984-01-01', '2000-12-31'),
    'Period_2': ('2001-01-01', '2017-12-31')
}

def get_subperiod_data(start_date, end_date):
    r_sub = r_all_wide.loc[start_date:end_date]
    x_sub = X_all.loc[start_date:end_date]
    
    min_obs = WINDOW_HIGH + N_LAGS + 100
    # Filter to eligible stocks with enough observations in this subperiod
    eligible = [st for st in r_sub.columns if r_sub[st].dropna().shape[0] >= min_obs]
    return r_sub, x_sub, eligible

In [27]:
# ── Tuning Functions ─────────────────────────────────────────────────────────
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))

def score_from_summary(sm):
    if sm is None:
        return np.nan
    tstat    = float(sm.get('kappa_tstat', np.nan))
    sel_rate = float(sm.get('avg_selection_rate', np.nan))
    kappa    = float(sm.get('kappa', np.nan))
    
    if not all(np.isfinite([tstat, sel_rate, kappa])):
        return np.nan
    if sel_rate < 0.003 or sel_rate > 0.20:
        return 0.0
        
    sel_reward   = np.exp(-((sel_rate - 0.03) / 0.02) ** 2)
    tstat_reward = sigmoid(tstat - 1.96)
    kappa_reward = sigmoid((kappa - 0.3) / 0.2)
    return 0.7 * tstat_reward + 0.2 * sel_reward + 0.1 * kappa_reward

def run_one_tuning(st, window_size, lam_base, r_sub, x_sub):
    r = r_sub[st].dropna()
    if len(r) < window_size + N_LAGS + 100:
        return None
        
    stock_vol = float(r.std())
    eff_lam   = lam_base * (stock_vol / TARGET_VOL)
    X = x_sub.loc[r.index]
    
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        try:
            res = estimate_single_config_fast(
                X=X, y=r, window_size=window_size,
                n_lags=N_LAGS, lambda_val=eff_lam, return_details=False
            )
            return res.get('summary', None)
        except Exception:
            return None

In [28]:
from tqdm.auto import tqdm

# ── Subperiod Optuna Tuning Execution ────────────────────────────────────────
best_params_per_period = {}

for period_name, (p_start, p_end) in tqdm(SUBPERIODS.items(), desc="Overall Progress (Periods)"):
    print(f'\n--- Tuning for {period_name} ({p_start} to {p_end}) ---')
    r_sub, x_sub, eligible = get_subperiod_data(p_start, p_end)
    print(f'Eligible stocks: {len(eligible)}')
    
    if len(eligible) == 0:
        print(f'Skipping {period_name} - no eligible stocks.')
        continue
        
    rng = np.random.default_rng(SEED)
    n_tune = max(1, int(len(eligible) * TUNE_FRAC))
    tune_stocks = list(rng.choice(eligible, size=n_tune, replace=False))
    
    trial_rng = np.random.default_rng(SEED + 1)
    trial_stocks = list(trial_rng.choice(tune_stocks, size=N_TRIALS, replace=True))
    
    def objective(trial):
        lam = trial.suggest_float('lambda', LAMBDA_LOW, LAMBDA_HIGH, log=True)
        s   = trial.suggest_int('window_size', WINDOW_LOW, WINDOW_HIGH)
        st  = trial_stocks[trial.number]
        sm  = run_one_tuning(st, s, lam, r_sub, x_sub)
        sc  = score_from_summary(sm)
        return float(sc) if (sc is not None and np.isfinite(sc)) else 0.0

    sampler = optuna.samplers.TPESampler(seed=SEED)
    study   = optuna.create_study(direction='maximize', sampler=sampler, study_name=f'cs_tuning_{period_name}')
    
    # --- NEW: Manually wrap the trial progress bar ---
    # `leave=False` makes the nested bar disappear when finished, keeping your notebook clean!
    with tqdm(total=N_TRIALS, desc=f"Trials ({period_name})", leave=False) as pbar:
        
        # This callback updates our custom progress bar every time a trial finishes
        def pbar_callback(study, trial):
            pbar.update(1)
        
        # Pass the callback and keep Optuna's native bar OFF
        study.optimize(
            objective, 
            n_trials=N_TRIALS, 
            callbacks=[pbar_callback], 
            show_progress_bar=False 
        )
    # -------------------------------------------------
    
    best = study.best_trial
    best_params_per_period[period_name] = {
        'lambda': best.params['lambda'],
        'window_size': int(best.params['window_size']),
        'eligible_stocks': eligible,
        'tune_stocks': tune_stocks  
    }
    
    print(f'Best for {period_name} → lambda={best.params["lambda"]:.3e}, '
          f'window={best.params["window_size"]}, score={best.value:.4f}')

Overall Progress (Periods):   0%|          | 0/2 [00:00<?, ?it/s]


--- Tuning for Period_1 (1984-01-01 to 2000-12-31) ---
Eligible stocks: 3248


Trials (Period_1):   0%|          | 0/200 [00:00<?, ?it/s]

Best for Period_1 → lambda=2.501e-03, window=256, score=0.8998

--- Tuning for Period_2 (2001-01-01 to 2017-12-31) ---
Eligible stocks: 3465


Trials (Period_2):   0%|          | 0/200 [00:00<?, ?it/s]

Best for Period_2 → lambda=1.805e-03, window=345, score=0.8061


This notebook estimates the model on two different subperiods: 1984-01-01 until 2000-12-31 and 2001-01-01 until 2017-12-31.

We tune the hyperparamters on each subperiod and then estimate the model seperatly in each subsample.

In [29]:
# ── Full Estimation Functions ────────────────────────────────────────────────
import re
_LAG1_RE = re.compile(r'^Lasso_(.+)_lag_1$')

def process_stock_subperiod(st, r_sub, x_sub, best_lam, best_window, stock_vol):
    r = r_sub[st].dropna()
    eff_lam = best_lam * (stock_vol / TARGET_VOL)

    if len(r) < best_window + N_LAGS + 100:
        return {'stock': st, 'failed': True, 'reason': 'too short'}

    X = x_sub.loc[r.index]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        try:
            res = estimate_single_config_fast(
                X=X, y=r, window_size=best_window,
                n_lags=N_LAGS, lambda_val=eff_lam, return_details=True
            )
        except Exception as e:
            return {'stock': st, 'failed': True, 'reason': str(e)}

    details = res.get('details')
    if details is None or details.empty:
        return {'stock': st, 'failed': True, 'reason': 'empty details'}

    summary = dict(res.get('summary') or {})
    summary['stock']      = st
    summary['eff_lambda'] = eff_lam
    summary['stock_vol']  = stock_vol

    topic_cols = {}
    for c in details.columns:
        m = _LAG1_RE.match(str(c))
        if m:
            topic_cols[m.group(1)] = c
    topics_found = sorted(topic_cols.keys())

    valid    = details['date'].notna() if 'date' in details.columns else pd.Series(True, index=details.index)
    sub      = details.loc[valid]
    date_arr = pd.to_datetime(sub['date']).to_numpy()
    beta_arr = sub[[topic_cols[t] for t in topics_found]].to_numpy(dtype=np.float32)
    r2in_arr = sub['lasso_r2_in'].to_numpy(dtype=np.float32)
    pred_arr = sub['prediction'].to_numpy(dtype=np.float32)
    targ_arr = sub['target'].to_numpy(dtype=np.float32)
    icpt_arr = sub['lasso_intercept'].to_numpy(dtype=np.float32)

    kappa      = summary.get('kappa', np.nan)
    intercept2 = summary.get('intercept', np.nan)
    stg2_pred  = np.full(len(pred_arr), np.nan, dtype=np.float32)
    if np.isfinite(kappa) and np.isfinite(intercept2) and len(pred_arr) > 1:
        try:
            alm = compute_alm_returns(pred_arr, kappa, intercept2)
            stg2_pred[1:] = alm
        except Exception:
            pass

    return {
        'stock': st, 'failed': False,
        'topics': topics_found, 'dates': date_arr,
        'betas': beta_arr, 'r2_in': r2in_arr,
        'predictions': pred_arr, 'targets': targ_arr,
        'intercepts': icpt_arr, 'stg2_preds': stg2_pred,
        'summary': summary,
    }

In [32]:
# ── Execute Estimation per Subperiod ─────────────────────────────────────────
for period_name, params in best_params_per_period.items():
    p_start, p_end = SUBPERIODS[period_name]
    
    r_sub, x_sub, eligible = get_subperiod_data(p_start, p_end)
    
    best_lam = params['lambda']
    best_win = params['window_size']
    tune_stocks = set(params['tune_stocks']) # Fetch the tuning stocks
    
    estimation_stocks = [st for st in eligible if st not in tune_stocks] 
    
    stock_vols = {st: float(r_sub[st].dropna().std()) for st in estimation_stocks}
    
    period_cache_dir = OUT_DIR / f'_cache_{period_name}'
    period_cache_dir.mkdir(parents=True, exist_ok=True)
    
    def cache_path(st):
        return period_cache_dir / f'{st}.pkl'
    def already_done(st):
        return cache_path(st).exists()
    def save_cache(result):
        with open(cache_path(result['stock']), 'wb') as f:
            pickle.dump(result, f, protocol=4)
            
    pending = [s for s in estimation_stocks if not already_done(s)]
    batches = [pending[i:i+BATCH_SIZE] for i in range(0, len(pending), BATCH_SIZE)]
    
    print(f'Eligible (Out-of-Sample): {len(estimation_stocks)} | Pending: {len(pending)} | Batches: {len(batches)}')
    
    for b_idx, batch in enumerate(batches):
        results = Parallel(n_jobs=N_JOBS, backend='loky')(
            delayed(process_stock_subperiod)(st, r_sub, x_sub, best_lam, best_win, stock_vols[st]) 
            for st in batch
        )
        for r in results:
            save_cache(r)
        print(f'  Batch {b_idx+1}/{len(batches)} finished.')
        
    # Load & Assemble HDF5
    all_results = []
    for st in eligible:
        p = cache_path(st)
        if p.exists():
            with open(p, 'rb') as f:
                all_results.append(pickle.load(f))
                
    ok_results = [r for r in all_results if not r['failed']]
    print(f'Assembling HDF5 for {period_name}. Usable: {len(ok_results)}/{len(all_results)}')
    
    all_date_strs = set()
    topics_ref = None
    for r in ok_results:
        for d in r['dates']:
            all_date_strs.add(str(pd.Timestamp(d))[:10])
        if topics_ref is None:
            topics_ref = r['topics']

    if not ok_results:
        continue

    date_strs = sorted(all_date_strs)
    date_to_i = {d: i for i, d in enumerate(date_strs)}
    n_stocks  = len(eligible)
    n_topics  = len(topics_ref)
    n_dates   = len(date_strs)
    stock_to_i = {s: i for i, s in enumerate(eligible)}
    
    T_betas   = np.full((n_stocks, n_topics, n_dates), np.nan, dtype=np.float32)
    T_r2in    = np.full((n_stocks, n_dates),           np.nan, dtype=np.float32)
    T_preds   = np.full((n_stocks, n_dates),           np.nan, dtype=np.float32)
    T_targets = np.full((n_stocks, n_dates),           np.nan, dtype=np.float32)
    T_icpts   = np.full((n_stocks, n_dates),           np.nan, dtype=np.float32)
    T_stg2    = np.full((n_stocks, n_dates),           np.nan, dtype=np.float32)
    summary_rows = []
    
    for r in ok_results:
        si = stock_to_i[r['stock']]
        for j, d in enumerate(r['dates']):
            k = date_to_i.get(str(pd.Timestamp(d))[:10])
            if k is None: continue
            T_betas[si, :, k]  = r['betas'][j]
            T_r2in[si, k]      = r['r2_in'][j]
            T_preds[si, k]     = r['predictions'][j]
            T_targets[si, k]   = r['targets'][j]
            T_icpts[si, k]     = r['intercepts'][j]
            T_stg2[si, k]      = r['stg2_preds'][j]
        summary_rows.append(r['summary'])
        
    for r in all_results:
        if r['failed']:
            summary_rows.append({'stock': r['stock'], 'failed': True, 'reason': r.get('reason', '')})

    summary_df = pd.DataFrame(summary_rows).set_index('stock')

    h5_path = OUT_DIR / f'betas_{period_name}.h5'
    with h5py.File(h5_path, 'w') as f:
        kw = dict(dtype='float32', compression='gzip', compression_opts=4)
        f.create_dataset('betas',               data=T_betas,   **kw)
        f.create_dataset('r2_in',               data=T_r2in,    **kw)
        f.create_dataset('predictions',         data=T_preds,   **kw)
        f.create_dataset('targets',             data=T_targets, **kw)
        f.create_dataset('lasso_intercepts',    data=T_icpts,   **kw)
        f.create_dataset('stage2_predictions',  data=T_stg2,    **kw)
        sd = h5py.string_dtype()
        
        # Explicitly convert lists to string format to satisfy h5py's strict typing
        stocks_str = [str(s) for s in eligible]
        topics_str = [str(t) for t in topics_ref]
        dates_str  = [str(d) for d in date_strs]
        
        f.create_dataset('stocks', data=np.array(stocks_str, dtype=sd))
        f.create_dataset('topics', data=np.array(topics_str, dtype=sd))
        f.create_dataset('dates',  data=np.array(dates_str,  dtype=sd))
        
    csv_path = OUT_DIR / f'summary_{period_name}.csv'
    summary_df.to_csv(csv_path)
    print(f'Saved HDF5 -> {h5_path} Tensor: {T_betas.shape}')
    print(f'Saved CSV  -> {csv_path}')
    
print('\nAll periods fully estimated!')

Eligible (Out-of-Sample): 2599 | Pending: 0 | Batches: 0
Assembling HDF5 for Period_1. Usable: 2599/2599
Saved HDF5 -> C:\Users\jonat\Lasso_paper\Results\Estimation\Subperiods\betas_Period_1.h5 Tensor: (3248, 180, 4025)
Saved CSV  -> C:\Users\jonat\Lasso_paper\Results\Estimation\Subperiods\summary_Period_1.csv
Eligible (Out-of-Sample): 2772 | Pending: 2772 | Batches: 56
  Batch 1/56 finished.
  Batch 2/56 finished.
  Batch 3/56 finished.
  Batch 4/56 finished.
  Batch 5/56 finished.
  Batch 6/56 finished.
  Batch 7/56 finished.
  Batch 8/56 finished.
  Batch 9/56 finished.
  Batch 10/56 finished.
  Batch 11/56 finished.
  Batch 12/56 finished.
  Batch 13/56 finished.
  Batch 14/56 finished.
  Batch 15/56 finished.
  Batch 16/56 finished.
  Batch 17/56 finished.
  Batch 18/56 finished.
  Batch 19/56 finished.
  Batch 20/56 finished.
  Batch 21/56 finished.
  Batch 22/56 finished.
  Batch 23/56 finished.
  Batch 24/56 finished.
  Batch 25/56 finished.
  Batch 26/56 finished.
  Batch 27/5

In [ ]:
import pandas as pd
import numpy as np
import h5py

# ── Compare Subperiod Estimations ─────────────────────────────────────────────

comparison_results = []

for period_name in SUBPERIODS.keys():
    h5_path = OUT_DIR / f'betas_{period_name}.h5'
    csv_path = OUT_DIR / f'summary_{period_name}.csv'
    
    if not h5_path.exists() or not csv_path.exists():
        print(f"Skipping {period_name}: Output files not found.")
        continue
        
    # --- 1. Load CSV Summary Data ---
    summary_df = pd.read_csv(csv_path)
    
    # Filter out failed estimation runs
    if 'failed' in summary_df.columns:
        valid_df = summary_df[summary_df['failed'] != True].copy()
    else:
        valid_df = summary_df.copy()
        
    n_stocks = len(valid_df)
    
    # Extract Kappa statistics (Handling potential column naming variations)
    tstat_col = 'kappa_tstat' if 'kappa_tstat' in valid_df.columns else 't_stat'
    kappa_col = 'kappa' if 'kappa' in valid_df.columns else 'kappa_est'
    
    sig_kappa_pct = np.nan
    median_kappa = np.nan
    
    if tstat_col in valid_df.columns:
        # Significant at 5% level (|t| > 1.96)
        sig_mask = valid_df[tstat_col].abs() > 1.96
        sig_kappa_pct = (sig_mask.sum() / n_stocks) * 100
        
        if kappa_col in valid_df.columns:
            median_kappa = valid_df.loc[sig_mask, kappa_col].median()

    # --- 2. Load HDF5 Tensor Data ---
    with h5py.File(h5_path, 'r') as f:
        betas = f['betas'][:]
        targets = f['targets'][:]
        preds = f['predictions'][:]
        stg2_preds = f['stage2_predictions'][:]
        r2_in_array = f['r2_in'][:]
        
    # Overall LASSO Selection Rate
    # Exclude NaNs from the tensor before calculating the non-zero mean
    valid_betas = betas[~np.isnan(betas)]
    selection_rate = (valid_betas != 0).mean() * 100
    
    # Mean In-Sample R2
    mean_r2_in = np.nanmean(r2_in_array)
    
    # --- 3. Compute Out-of-Sample R2 ---
    # The paper uses the zero-forecast (0) as the benchmark for returns
    
    # Stage 1 (LASSO) OOS R2
    valid_mask_stg1 = ~np.isnan(targets) & ~np.isnan(preds)
    if np.any(valid_mask_stg1):
        mse_model_stg1 = np.sum((targets[valid_mask_stg1] - preds[valid_mask_stg1])**2)
        mse_bench_stg1 = np.sum((targets[valid_mask_stg1])**2) 
        oos_r2_stg1 = 1 - (mse_model_stg1 / mse_bench_stg1)
    else:
        oos_r2_stg1 = np.nan
        
    # Stage 2 (ALM) OOS R2
    valid_mask_stg2 = ~np.isnan(targets) & ~np.isnan(stg2_preds)
    if np.any(valid_mask_stg2):
        mse_model_stg2 = np.sum((targets[valid_mask_stg2] - stg2_preds[valid_mask_stg2])**2)
        mse_bench_stg2 = np.sum((targets[valid_mask_stg2])**2)
        oos_r2_stg2 = 1 - (mse_model_stg2 / mse_bench_stg2)
    else:
        oos_r2_stg2 = np.nan

    # --- 4. Append to Comparison ---
    comparison_results.append({
        'Period': period_name,
        'N Stocks': n_stocks,
        'Selection Rate (%)': selection_rate,
        'In-Sample R2': mean_r2_in,
        'Stage 1 OOS R2': oos_r2_stg1,
        'Stage 2 OOS R2': oos_r2_stg2,
        'Significant Kappa (%)': sig_kappa_pct,
        'Median Kappa (Sig)': median_kappa
    })


SUBPERIOD ESTIMATION COMPARISON (LLM Narrative vs. Mechanical Sparsity)
          N Stocks  Selection Rate (%)  In-Sample R2  Stage 1 OOS R2  Stage 2 OOS R2  Significant Kappa (%)  Median Kappa (Sig)
Period                                                                                                                         
Period_1      2599              2.2986        0.0260         -0.0160         -0.0020                11.0042              0.2833
Period_2      2772              4.7354        0.0386         -0.0337         -0.0014                 9.1270              0.1715


In [ ]:
# ── Display Results ───────────────────────────────────────────────────────────

comp_df = pd.DataFrame(comparison_results).set_index('Period')

print("\n" + "="*90)
print("SUBPERIOD ESTIMATION COMPARISON (Period 1: 1984-2000; Period 2: 2001-2017)")
print("="*90)
# Round to 4 decimals for cleaner console output
print(comp_df.round(4).to_string())
print("="*90)


SUBPERIOD ESTIMATION COMPARISON (Period 1: 1984-2000; Period 2: 2001-2017)
          N Stocks  Selection Rate (%)  In-Sample R2  Stage 1 OOS R2  Stage 2 OOS R2  Significant Kappa (%)  Median Kappa (Sig)
Period                                                                                                                         
Period_1      2599              2.2986        0.0260         -0.0160         -0.0020                11.0042              0.2833
Period_2      2772              4.7354        0.0386         -0.0337         -0.0014                 9.1270              0.1715
